# Detection of Suicidal Ideation in Arabic Microblogging
**End-to-end Colab pipeline** — clones the project from GitHub, pulls the dataset from Google Drive, trains baselines + MARBERT, and renders comparison plots.

Before you run:
1. **Runtime → Change runtime type → GPU** (T4 is enough for MARBERT).
2. Edit the two variables in cell **2** (`GITHUB_REPO`, `DATA_PATH`).

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure (edit these two lines)

In [ ]:
# Your repo (HTTPS) — change to yours
GITHUB_REPO = 'https://github.com/<your-user>/NLP-DSIMC.git'
GIT_BRANCH  = 'main'

# Path to the dataset on your Google Drive
DATA_PATH   = '/content/drive/MyDrive/NLP/Arabic Suicidal Dataset.xlsx'

# (Optional) override these only if your column names are unusual
TEXT_COL    = ''   # leave empty for auto-detect
LABEL_COL   = ''

## 3. Clone the repo

In [ ]:
import os, shutil, subprocess
os.chdir('/content')
if os.path.exists('NLP-DSIMC'):
    shutil.rmtree('NLP-DSIMC')
subprocess.run(['git', 'clone', '-b', GIT_BRANCH, GITHUB_REPO, 'NLP-DSIMC'], check=True)
os.chdir('/content/NLP-DSIMC')
!ls -la

## 4. Install dependencies

In [ ]:
!pip -q install -r requirements.txt

## 5. Sanity-check the dataset path

In [ ]:
import os
assert os.path.exists(DATA_PATH), f'NOT FOUND: {DATA_PATH}'
print('OK ->', DATA_PATH)

## 6. EDA (plots saved to `artifacts/eda/`)

In [ ]:
extra = (f' --text-col "{TEXT_COL}"' if TEXT_COL else '') + \
        (f' --label-col "{LABEL_COL}"' if LABEL_COL else '')
!python RUN.py eda --data "$DATA_PATH" $extra

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('artifacts/eda/*.png')):
    print(p); display(Image(p))

## 7. Train baselines (TF-IDF + SVM/LogReg/RF)

In [ ]:
!python RUN.py train_baseline --data "$DATA_PATH" $extra

## 8. Fine-tune MARBERT (FP16 if GPU available)

In [ ]:
!python RUN.py train_marbert --data "$DATA_PATH" --fp16 --epochs 3 --batch-size 16 $extra

## 9. Compare every trained model on the same test split

In [ ]:
!python RUN.py compare --data "$DATA_PATH" $extra

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('artifacts/plots/*.png')):
    print(p); display(Image(p))

## 10. (Optional) Persist results back to Drive

In [ ]:
import os, shutil, time
out = f'/content/drive/MyDrive/NLP/artifacts_{time.strftime("%Y%m%d_%H%M")}'
shutil.copytree('artifacts', out)
print('Copied artifacts to:', out)

## 11. Inference demo

In [ ]:
!python RUN.py predict --text "حسيت اني تعبت من كل شي ومالي فايده"